<font color="#D35400" size='6px'><b>Video Semantic Edge (Boundary Tracking)</b></font>

<font color="#D35400" size='4px'><b>The Pipeline Architecture</b></font>

The system operates in a high-precision morphological pipeline designed to extract and track the boundaries of semantic objects across time, creating "moving sketches" or neon-like analytical visualizations.



<font color="#D35400" size='4px'><b>Stage I: Scene Understanding & Class Discovery (Frame 0)</b></font>
Before extraction begins, the system identifies the semantic entities present in the opening frame.

**Automatic Discovery (LLM/BLIP):** The first frame is analyzed by a Vision-Language Model.
* <font color="#884EA0"><i>LLM Mode:</i></font> Uses **Async Parallel Processing** to analyze image crops, identifying complex boundaries and specific objects.
* <font color="#884EA0"><i>BLIP Mode:</i></font> Uses local captioning to find standard objects (e.g., "car", "pedestrian").

**Refinement:** The discovered nouns are cleaned via NLP to remove abstract concepts and ensure only physical objects with clear boundaries are targeted.

<font color="#D35400" size='4px'><b>Stage II: SAM 3 Tracker Initialization</b></font>
The system leverages the **SAM 3 Video Predictor** to establish temporal consistency.

**Session Management:** A unique "Inference Session" is created to maintain object identity across the video duration.

**Prompting Frame 0:** Semantic targets are converted into text prompts for SAM 3, which generates high-fidelity initial masks for every object instance.

<font color="#D35400" size='4px'><b>Stage III: Temporal Propagation (The "Boundary" Memory)</b></font>
Instead of calculating edges from scratch on every frame, the system propagates semantic knowledge through time.

**Memory Bank:** SAM 3 builds a visual memory of the object's features to track its boundary even through rotation or deformation.

**Stream Tracking:** As the video advances, the model "propels" the object masks forward, ensuring that the resulting edges belong to the same persistent object throughout the scene.

<font color="#D35400" size='4px'><b>Stage IV: Morphological Edge Extraction</b></font>
The tracked masks are mathematically converted into thin outlines in real-time.

**Morphological Gradient:** For every frame, the binary masks are processed using `cv2.morphologyEx`.
* <font color="#884EA0"><i>Gradient Calculation:</i></font> Defined as the difference between **Dilation** (expanding) and **Erosion** (shrinking) of the mask.
* <font color="#884EA0"><i>Precision:</i></font> This isolates only the boundary pixels, ignoring internal texture noise and background clutter.



<font color="#D35400" size='4px'><b>Stage V: Visualization & Video Rendering</b></font>
The final step reconstructs the video with rich analytical overlays.

**Stylized Rendering:**
* <font color="#884EA0"><i>B&W Sketch:</i></font> Renders a high-contrast binary video showing white semantic outlines on a black background.
* <font color="#884EA0"><i>Colored Edges:</i></font> Outlines are color-coded by class (e.g., "Vehicle" = Blue, "Person" = Red) for semantic clarity.

**Data Persistence:** The processed edge videos and compressed mask metadata are saved to the `notebook_outputs` directory.

<font color="#2E86C1" size='6px'>Environment Initialization & Repository Setup</font>

In [ ]:
COLAB = True

In [ ]:
import os
import sys

if COLAB:
    # Clone Repository 
    if not os.path.exists("sam3-toolkit"):
        print("--- Cloning repository...")
        !git clone https://github.com/MrKGhasemi/sam3-toolkit.git
    else:
        print("--- Repository already exists.")

    # Auto-Detect Paths 
    repo_root = os.path.abspath("sam3-toolkit")
    sam3_install_path = None
    practical_path = None

    for root, dirs, files in os.walk(repo_root):
        if ("sam3" in dirs or "sam3_main" in root or root == repo_root):
            if sam3_install_path is None or len(root) < len(sam3_install_path):
                sam3_install_path = root

        if "semantic_edge.py" in files:
            practical_path = root

    if not sam3_install_path or not practical_path:
        raise FileNotFoundError(
            "     --- Critical paths (sam3 or practical) not found.")

    print(f"Found SAM 3 Library at: {sam3_install_path}")
    print(f"Found Project Code at: {practical_path}")

    # Install Dependencies 
    print("--- Installing Python packages...")
    !{sys.executable} -m pip install -q opencv-python matplotlib scikit-learn transformers spacy openai gdown mega.py huggingface_hub
    !{sys.executable} -m pip install -q einops decord pycocotools

    # Install SAM 3 
    print("--- Installing SAM 3 Library...")
    !{sys.executable} -m pip install -e "{sam3_install_path}"

    # Configure Working Directory 
    os.chdir(practical_path)
    if practical_path not in sys.path:
        sys.path.insert(0, practical_path)
    print(f"--- Working directory set to: {os.getcwd()}")

    # Download Model Weights
    weights_dir = "weights"
    weights_path = os.path.join(weights_dir, "sam3.pt")
    os.makedirs(weights_dir, exist_ok=True)

    # If the model is Private/Gated, you MUST provide a token.
    # Get it here: https://huggingface.co/settings/tokens
    HF_REPO_ID = "facebook/sam3" 
    HF_FILENAME = "sam3.pt"               
    HF_TOKEN = "HF_TOKEN"  # Replace with your actual token or set to None

    if not os.path.exists(weights_path):
        print(f"--- Starting download...")

        try:
            from huggingface_hub import hf_hub_download
            print("--- Connecting to Hugging Face Hub...")
            downloaded_file = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                local_dir=weights_dir,
                token=HF_TOKEN,
                local_dir_use_symlinks=False  # Ensure we get the actual file, not a symlink
            )
            if os.path.basename(downloaded_file) != "sam3.pt":
                os.rename(downloaded_file, weights_path)

            print("--- Download attempt finished.")

        except Exception as e:
            print(f"      --- Error downloading: {e}")
            print("           Tip: If using Hugging Face private repo, ensure HF_TOKEN is set.")

    else:
        print("--- Weights file already exists.")

    # Verify File Integrity 
    if os.path.exists(weights_path):
        final_size = os.path.getsize(weights_path) / (1024 * 1024)
        print(f"--- Final File Size: {final_size:.2f} MB")
        if final_size < 2000:
            print(
                "      --- WARNING: File seems too small (<2GB). It might be corrupt or a placeholder.")
    else:
        print("      --- Error: File not found after download.")

    # Download SpaCy Model 
    print("--- Checking SpaCy model...")
    !{sys.executable} -m spacy download en_core_web_lg

    # Verify Import 
    print("\n--- Verifying imports...")
    repo_root = os.path.abspath("/sam3-toolkit")

    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
        print(f"--- Added '{repo_root}' to sys.path")

    practical_dir = os.path.join(repo_root, "practical")
    if practical_dir not in sys.path:
        sys.path.insert(0, practical_dir)
        print(f"--- Added '{practical_dir}' to sys.path")

    # Verify all dependencies
    print("\n--- Retrying imports...")
    try:
        import sam3
        print("--- 'sam3' library loaded!")
    except ImportError as e:
        print(f"      --- Failed to load sam3: {e}")
        print("       --- Attempting hard install fix...")
        !pip install "{repo_root}"
        import sam3

    try:
        import models
        print("--- 'models.py' loaded!")
    except ImportError as e:
        print(f"      --- Failed to load models: {e}")

    !pip uninstall numpy -y
    !{sys.executable} -m pip install "numpy<2.0"

    # Check GPU
    import torch
    print(f"--- GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"--- GPU Name: {torch.cuda.get_device_name(0)}")

    if os.path.exists(practical_dir):
        os.chdir(practical_dir)
        print(f"--- Current Directory: {os.getcwd()}")


---

<font color="#117A65" size='6px'>Loading Core Modules & Dependencies</font>

<font color="#D68910" size='4px'>*semantic_edge:*</font> Contains the main pipeline for generate edge and boundary of objects.

<font color="#D68910" size='4px'>*visualization:*</font> Handles the drawing of bounding boxes, masks, and legends.



In [ ]:
import torch
import os
import sys
root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root)

if os.path.basename(os.getcwd()) == "ipynb":
    os.chdir(root)

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
import semantic_edge
import torch
import torchvision
import visualization
import utils

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

In [ ]:
import numpy
print(numpy.__version__) 
# process is Not capable with numpy 2.0.0 or higher

```Python
# From configs.py 
LLM_CAPTION_MODEL_NAME = "gpt-5-chat"
SAM3_CONF_OBJECT_COUNTING_VIDEO_THRESHOLD = 0.2
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

args = MockArgs(mode="blip", output_dir=output_directory)

# for LLM mode:
# args = MockArgs(
#     mode="llm",
#     output_dir=output_directory,
#     api_key="api_key",
#     base_url="base_url"
# )

---

<font color="#D35400" size='6px'>Video Execution: Semantic Edge Tracking</font>

This cell executes the Temporal Edge Pipeline. By calling <font color="#D35400" size='4px'>semantic_edge.semantic_edge_video</font>, the system reuses the high-performance tracking engine to generate consistent object boundaries across the entire video duration.

The function orchestrates a sophisticated boundary-tracking logic:

<font color="#D35400" size='4px'><i>Integrated Wrapper:</i></font> The script avoids redundant logic by acting as a style-wrapper around <font color="#D35400"><i>count_objects_video</i></font>. This ensures that the objects being "sketched" are the same high-confidence instances identified by the counting module.

<font color="#D35400" size='4px'><i>Temporal Propagation:</i></font> Using the SAM 3 Video Predictor, the system "propels" semantic masks from frame to frame. This allows the edge maps to remain stable and flicker-free, even when objects rotate or the camera moves rapidly.

<font color="#D35400" size='4px'><i>Refined Parameterization:</i></font> The function injects a <font color="#D35400"><i>thickness</i></font> parameter into the pipeline, which dictates the kernel size for the eventual morphological gradient, allowing for customization between "fine-line" and "bold-border" sketches.

```python
# Wrapper logic from semantic_edge.py
def semantic_edge_video(video_path, args, thickness=2):
    # 1. Reuse existing video tracker for object consistency
    output_information = count_objects_video(video_path, args, task_type="automatic")
    
    # 2. Inject edge-specific style parameters
    output_information["thickness"] = thickness
    
    # 3. Call the specialized Edge Visualizer
    visualization.semantic_edges_visualizations(output_information)
```    

In [ ]:
VIDEO_PATH = "videos/walking.mp4"

In [ ]:
output_information = semantic_edge.semantic_edge_video(VIDEO_PATH, args, thickness=2)

<font color="#8E44AD" size='6px'>Video Visualization: Morphological Rendering</font>

The final stage of the pipeline transforms raw tracking data into artistic or analytical edge videos. The system uses mathematical morphology to isolate object perimeters from their solid masks.

The rendering process involves three key layers:

<font color="#8E44AD" size='4px'><i>Morphological Gradient:</i></font> For every frame, the system processes masks using <font color="#8E44AD"><i>cv2.morphologyEx</i></font>. It calculates the difference between the Dilation and Erosion of each mask, mathematically isolating the boundary pixels (contours) while ignoring the object's interior.

<font color="#8E44AD" size='4px'><i>Class-Aware Colorization:</i></font> The system maps the unique boundary of each object to its specific class color. This creates a "Semantic Outline" where, for example, all people are outlined in red while vehicles are outlined in blue, maintaining visual context throughout the scene.

<font color="#8E44AD" size='4px'><i>Dual-Stream Encoding:</i></font> The processed frames are written to two separate files using <font color="#8E44AD"><i>cv2.VideoWriter</i></font>: a high-contrast B&W map (ideal for structural analysis) and a Color map (ideal for semantic presentation).

```python
# Per-frame edge calculation from visualization.py
kernel = np.ones((thickness, thickness), np.uint8)

for obj in frame_objs:
    mask = obj['mask'].astype(np.uint8) * 255
    # Calculate Morphological Gradient (Edges)
    edges = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, kernel)
    
    # Accumulate into BW and Color canvases
    frame_edge_bw = cv2.add(frame_edge_bw, edges)
    frame_edge_col[edges > 0] = class_color_bgr
```    

In [ ]:
visualization.semantic_edges_visualizations(output_information)

<font color="#2874A6" size='6px'>Universal Data Serialization</font>

This cell executes the Data Compression & Archiving step. Regardless of the specific task (Counting, Segmentation, or Redaction), the results—often containing thousands of high-resolution binary masks—are massive. This function efficiently packs them for storage.

The function <font color="#2874A6"><i>utils.compress_and_save_output_information</i></font> applies a rigorous optimization pipeline:

<font color="#2874A6" size='4px'><i>RLE Compression:</i></font> Instead of saving dense boolean arrays (which consume huge disk space), the function converts every segmentation mask into <font color="#2874A6"><i>Run-Length Encoding (RLE)</i></font> using <font color="#2874A6"><i>pycocotools</i></font>. This compresses the data by storing counts of consecutive pixels rather than the pixels themselves.

<font color="#2874A6" size='4px'><i>Memory Optimization:</i></font> Before encoding, the binary masks are converted to <font color="#2874A6"><i>Fortran-contiguous arrays</i></font> via <font color="#2874A6"><i>np.asfortranarray</i></font>. This specific memory layout is required for high-speed C-based encoding operations, ensuring the compression process doesn't bottleneck the pipeline.

<font color="#2874A6" size='4px'><i>Structured Pickling:</i></font> The simplified, compressed dictionary is serialized using Python's <font color="#2874A6"><i>pickle</i></font> module. This preserves the complex nested structure (frames -> objects -> metadata) in a single file that can be instantly reloaded for analysis later.

In [ ]:
save_path = os.path.join(output_directory, "semantic_edge_output_walking_compressed.pkl")

utils.compress_and_save_output_information(output_information, save_path)

---

<font color="#3c983c" size='6px'>LLM-Enhanced Reasoning</font>

This cell activates the LLM Mode. By simply setting <font color="#3c983c" size='4px'>args.mode='llm'</font>, bypass the standard local models and instead use a Multimodal LLM to get a much deeper understanding of the scene.

This function runs on a fast Async Pipeline:

<font color="#CB4335" size='4px'><i>Asyncio Parallelism:</i></font> unlike BLIP (which works sequentially), the function <font color="#CB4335"><i>get_classes_llm</i></font> chops the image into multiple detailed crops using <font color="#CB4335" size='4px'><i>visualization.get_image_crops*</i></font>. It then sends API requests for all those crops at the exact same time, analyzing the whole image in parallel rather than waiting.

<font color="#CB4335" size='4px'><i>Visual Reasoning:</i></font> Instead of a generic "caption", instruct the model to focus strictly on describing physical objects. This allows the system to spot subtle or complex items that standard BLIP models usually miss.

<font color="#CB4335" size='4px'><i>Intelligent Parsing:</i></font> The raw descriptions from the vision model are collected and sent to a second "Parser" LLM. This step, managed by <font color="#CB4335"><i>utils.search_engine_clean_nouns</i></font>, cleans up the text and extracts a neat list of target nouns that act as precise prompts for SAM3.

In [ ]:
VIDEO_PATH = "videos/football.mp4"

In [ ]:
# for LLM mode:
args = MockArgs(
    mode="llm",
    output_dir=output_directory,
    api_key="api_key",
    base_url="base_url"
)

os.makedirs(args.output, exist_ok=True)

In [ ]:
output_information2 = semantic_edge.semantic_edge_video(VIDEO_PATH, args, thickness=2)

In [ ]:
visualization.semantic_edges_visualizations(output_information2)

In [ ]:
save_path = os.path.join(output_directory, "semantic_edge_output_football_compressed.pkl")

utils.compress_and_save_output_information(output_information2, save_path)

---

In [ ]:
object_counting_info = os.path.join(output_directory, "object_counting_output_football_compressed.pkl")

In [ ]:
loaded_information = utils.load_and_decompress_output_information(object_counting_info)

In [ ]:
visualization.semantic_edges_visualizations(loaded_information)